# 🚀 RepoTopology: Train Gemma 4 with 4-Bit QLoRA in Google Colab
### *Google - The Gemma 4 Developer Agent Competition*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/catlikeflyer/repotopology/blob/main/notebooks/01_train_gemma4_qlora.ipynb)

This notebook trains a PEFT/LoRA adapter on the **RepoTopology** AST traversal and context-pruning trajectories.

**Colab Runtime Setup:**
1. Navigate to **Runtime** > **Change runtime type**
2. Select **A100 GPU** (recommended for 31B) or **L4 / T4 GPU** (for E4B)
3. Set **RAM** to **High-RAM**
4. Click **Runtime** > **Run all**

## 1. Environment Setup & Hardware Inspection

In [ ]:
# Install dependencies
!pip install -q -U transformers peft trl datasets accelerate bitsandbytes

import os
import sys
import json
import shutil
from pathlib import Path
import torch
import torch.nn as nn
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.models.gemma4 import modeling_gemma4
from trl import SFTConfig, SFTTrainer
from google.colab import files

# Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
    print(f"[GPU Detected] {gpu_name} with {vram_gb} GB VRAM")
else:
    print("[WARNING] No GPU detected! Please change runtime type to GPU (A100 or L4).")

# Fix Gemma4ClippableLinear for PEFT
modeling_gemma4.Gemma4ClippableLinear.__bases__ = (nn.Linear,)

## 2. Mount Google Drive & Locate Dataset

In [ ]:
# Mount Google Drive for automatic persistence
try:
    from google.colab import drive
    drive.mount('/content/drive')
    has_drive = True
    print("[SUCCESS] Google Drive mounted successfully at /content/drive")
except Exception as e:
    has_drive = False
    print(f"[INFO] Drive mount skipped: {e}. Output will be saved to local Colab session.")

# Locate sft_trajectories.jsonl from Drive, local directory, or prompt user
DRIVE_DATA_DIR = "/content/drive/MyDrive/repology_data"
TRAJ_FILE = None

candidate_paths = [
    f"{DRIVE_DATA_DIR}/sft_trajectories.jsonl",
    "./data/sft_trajectories.jsonl",
    "sft_trajectories.jsonl",
    "/content/sft_trajectories.jsonl",
]

for p in candidate_paths:
    if os.path.exists(p):
        TRAJ_FILE = p
        break

if TRAJ_FILE:
    print(f"[SUCCESS] Found trajectory dataset at: {TRAJ_FILE}")
else:
    print("[ACTION REQUIRED] 'sft_trajectories.jsonl' not found in Drive or session files.")
    print("Uploading 'sft_trajectories.jsonl' now:")
    uploaded = files.upload()
    for fname in uploaded:
        if fname.endswith(".jsonl"):
            TRAJ_FILE = fname
            break

## 3. Load Tokenizer & Format Trajectories

In [ ]:
MODEL_ID = "google/gemma-4-31b-it"  # Or "google/gemma-4-e4b-it"
OUTPUT_DIR = "./adapters/main_lora"
DRIVE_BACKUP = "/content/drive/MyDrive/main_lora_31b.zip"

print(f"Loading Tokenizer for {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

formatted_data = []
with open(TRAJ_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            record = json.loads(line)
            msgs = record.get("messages", [])
            if msgs:
                text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
                formatted_data.append({"text": text})

print(f"Loaded {len(formatted_data)} decision trajectory samples.")
dataset = Dataset.from_list(formatted_data)

## 4. 4-Bit NF4 Quantization & Model Loading

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_ID} into VRAM with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

# Unwrap ClippableLinear layers for PEFT compatibility
for name, module in model.named_modules():
    for child_name, child in module.named_children():
        if hasattr(child, 'linear') and 'ClippableLinear' in child.__class__.__name__:
            setattr(module, child_name, child.linear)

model = prepare_model_for_kbit_training(model)

## 5. Configure LoRA & Train with TRL SFTTrainer

In [ ]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

training_args = SFTConfig(
    output_dir="./checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1.5e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    logging_steps=10,
    save_strategy="no",
    gradient_checkpointing=True,
    bf16=True,
    report_to="none",
    max_length=1024,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)

print("Starting QLoRA Fine-Tuning...")
trainer.train()

## 6. Save Adapter & Export to Google Drive / Local Download

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Saving adapter weights to {OUTPUT_DIR}...")
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Update adapter_config.json with competition alias
config_path = f"{OUTPUT_DIR}/adapter_config.json"
if os.path.exists(config_path):
    with open(config_path, "r") as f:
        cfg = json.load(f)
    cfg["base_model_name_or_path"] = "google/gemma-4-31b-it-qat-w4a16-ct"
    with open(config_path, "w") as f:
        json.dump(cfg, f, indent=2)

# Create zip archive
!zip -r /content/main_lora_31b.zip ./adapters/main_lora/

if has_drive:
    os.makedirs(os.path.dirname(DRIVE_BACKUP), exist_ok=True)
    shutil.copy2("/content/main_lora_31b.zip", DRIVE_BACKUP)
    print(f"[SUCCESS] Saved backup to Google Drive: {DRIVE_BACKUP}")

print("[SUCCESS] Triggering browser download of main_lora_31b.zip...")
files.download("/content/main_lora_31b.zip")